# 🔥 BurnScan · AIIMS — Colab Launcher
Clones both repos, installs deps, starts the FastAPI backend + serves the frontend, all via a public ngrok URL.

**Run cells in order.**

In [ ]:
# ── CELL 1 · Install system deps ──────────────────────────────────────────
!pip install -q fastapi uvicorn[standard] python-multipart \
               opencv-python-headless scikit-image matplotlib \
               Pillow pyngrok requests
print('✅ Packages installed.')

In [ ]:
# ── CELL 2 · Clone repos ──────────────────────────────────────────────────
# Replace with your actual GitHub usernames / repo names
CORE_REPO    = 'https://github.com/YOUR_USERNAME/burnscan-core.git'
WEBSITE_REPO = 'https://github.com/YOUR_USERNAME/burnscan-website.git'

!git clone --depth 1 {CORE_REPO}    burnscan-core    2>&1 | tail -3
!git clone --depth 1 {WEBSITE_REPO} burnscan-website 2>&1 | tail -3
print('✅ Repos cloned.')

In [ ]:
# ── CELL 3 · Set ngrok auth token ─────────────────────────────────────────
# Sign up free at https://dashboard.ngrok.com/signup
# Copy your token from https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_TOKEN = 'PASTE_YOUR_NGROK_TOKEN_HERE'   # ← replace

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
print('✅ ngrok token set.')

In [ ]:
# ── CELL 4 · Start FastAPI backend ────────────────────────────────────────
import subprocess, threading, time, requests, sys, os
from pyngrok import ngrok

API_PORT = 8000

def run_api():
    env = os.environ.copy()
    env['PYTHONPATH'] = 'burnscan-core:' + env.get('PYTHONPATH', '')
    subprocess.run(
        [sys.executable, '-m', 'uvicorn', 'main:app',
         '--host', '0.0.0.0', '--port', str(API_PORT)],
        cwd='burnscan-website/api',
        env=env,
    )

t = threading.Thread(target=run_api, daemon=True)
t.start()

print('⏳ Waiting for API to start…')
for _ in range(30):
    try:
        r = requests.get(f'http://localhost:{API_PORT}/api/health', timeout=2)
        if r.status_code == 200:
            print('✅ API is healthy:', r.json())
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print('⚠️  API did not start in time — check logs above.')

api_url = ngrok.connect(API_PORT)
print(f'\n🔌 API public URL: {api_url}')

In [ ]:
# ── CELL 5 · Patch frontend with API URL + serve it ───────────────────────
# Reads index.html, replaces the API_BASE localhost fallback with the ngrok
# API URL so the browser can reach the backend from any device.

import http.server, threading, os
from pyngrok import ngrok as _ngrok

FRONTEND_DIR = 'burnscan-website/frontend'
FRONTEND_PORT = 8080

# Patch index.html
html_path = os.path.join(FRONTEND_DIR, 'index.html')
with open(html_path, 'r') as f:
    html = f.read()

api_base = str(api_url).rstrip('/')   # e.g. https://xxxx.ngrok-free.app
html = html.replace(
    'const API_BASE = window.location.hostname === "localhost"',
    f'const API_BASE = "{api_base}" || window.location.hostname === "__never__"'
)
with open(html_path, 'w') as f:
    f.write(html)

# Serve with Python's built-in HTTP server
os.chdir(FRONTEND_DIR)
handler = http.server.SimpleHTTPRequestHandler
httpd   = http.server.HTTPServer(('', FRONTEND_PORT), handler)
tfe     = threading.Thread(target=httpd.serve_forever, daemon=True)
tfe.start()
os.chdir('/content')   # back to colab root

fe_url = _ngrok.connect(FRONTEND_PORT)

print()
print('=' * 65)
print(f'🔥  BurnScan is LIVE')
print(f'    Frontend : {fe_url}')
print(f'    API      : {api_url}')
print('=' * 65)
print()
print('Login credentials:')
print('  dr.sharma  /  aiims2024   (Doctor)')
print('  dr.mehta   /  burns123    (Resident)')
print()
print('Keep this cell running. Both URLs stay alive while Colab is active.')

In [ ]:
# ── CELL 6 (optional) · Quick API smoke test ──────────────────────────────
# Creates a tiny test image and sends it through the pipeline.
import numpy as np, cv2, requests
from io import BytesIO

# Synthetic burn-like image
img = np.zeros((200, 200, 3), dtype=np.uint8)
img[:, :]      = [120, 160, 190]   # skin-tone bg (BGR)
img[60:150, 60:150] = [20, 50, 160]  # red burn area

_, buf = cv2.imencode('.jpg', img)
files = {'file': ('test.jpg', BytesIO(buf.tobytes()), 'image/jpeg')}
data  = {'k': 10}

r = requests.post(f'http://localhost:{API_PORT}/api/analyse', files=files, data=data)
j = r.json()
print('Status  :', j['status'])
print('Degree  :', j['classification']['degree'])
print('Confidence:', j['classification']['confidence'])
print('TBSA %  :', j['classification']['tbsa_pct'])
print('Grid keys:', list(j['grids'].keys()))
print('✅ API smoke test passed.')

In [ ]:
# ── CELL 7 (optional) · Shut down ─────────────────────────────────────────
from pyngrok import ngrok
ngrok.kill()
print('✅ All tunnels closed.')